### Normalize and correct sgRNA Log Fold Change data
#### adapted from Fortin et al, 2019
#### by Stefanus Bernard

In [ ]:
import pandas as pd
from normalize_lfc_utils import *

#### Process Avana LFC and library data

In [ ]:
lfc_data = pd.read_csv("../../data/sgrna_lfc_data/avana_data/AvanaLogfoldChange_25Q2.csv")
lfc_data = lfc_data.rename(columns={lfc_data.columns[0]: 'spacer'})

# select specific columns
# lfc_data = lfc_data.iloc[:, :3]
lfc_data.head(25)

In [ ]:
# how many cell lines in Avana DepMap ?
# There are 1117 cell lines from Avana DepMap
# get first token before underscore for each column (skip the 'spacer' column)
col_prefixes = pd.Series(lfc_data.columns[1:]).str.extract(r'^(.*?)(?=Rep)')[0]

# number of unique prefixes (i.e. unique cell lines)
n_cell_lines = col_prefixes.nunique()
print("Number of unique prefixes (cell lines):", n_cell_lines)

# show the unique prefixes (first 50 for brevity) and counts
print(col_prefixes.value_counts().head(50))
col_prefixes.unique()[:50]
len(col_prefixes.unique())

In [ ]:
library_data = pd.read_csv("../../data/library_data/restricted_library/avana_library.tsv", sep ="\t", header = None)
library_data.columns = ['sgRNA', 'spacer', 'gene']
library_data.shape

In [ ]:
lfc = pd.merge(lfc_data, library_data, how="left", on="spacer").set_index(['sgRNA', 'spacer', 'gene']).sort_values(by="gene")
lfc

In [ ]:
# normalize data by median
lfc_norm = lfc.apply(normalize_column, axis=0)

In [ ]:
lfc_norm_scaled, list_common_essential = scale_essential(lfc, '../../data/sgrna_lfc_data/constitutive_core_essential_hart_2014.csv')
sanity_check_scale_essential(lfc_norm, lfc_norm_scaled, list_common_essential)

In [ ]:
lfc_norm_scaled_avg = mean_row(lfc_norm_scaled)
lfc_norm_scaled_avg = lfc_norm_scaled_avg.reset_index()
lfc_norm_scaled_avg

In [ ]:
lfc_norm_scaled_avg.to_csv("../../data/sgrna_lfc_data/output_normalized/avana_normalized_lfc.csv", index=False)